# ARC Prize 2026 — ARC-AGI-2 baseline submission

This notebook generates `submission.json` for the **ARC Prize 2026 (ARC-AGI-2)**
code competition using the dependency-free `arc_agi2` baseline.

**Code-competition constraints honoured here**
- No internet access required (pure standard-library Python).
- Writes `submission.json` to the working directory.
- Runs in well under the 12-hour CPU/GPU limit.

**Setup**: upload the `arc_prize_2026/arc_agi2` package directory as a Kaggle
Dataset (e.g. named `arc-agi2-baseline`) and attach it to this notebook, or
paste the package into a utility script. The next cell locates it on
`sys.path` automatically.

In [ ]:
import sys, glob, json
from pathlib import Path

# --- Locate the arc_agi2 package -------------------------------------------
# Attach the package as a Kaggle Dataset; we search the usual mount points.
PACKAGE_CANDIDATES = [
    "/kaggle/input/arc-agi2-baseline",
    "/kaggle/input/arc-agi2-baseline/arc_prize_2026",
    "/kaggle/usr/lib/arc_agi2",
    ".",
]
for cand in PACKAGE_CANDIDATES:
    if (Path(cand) / "arc_agi2" / "__init__.py").exists():
        sys.path.insert(0, cand)
        print("arc_agi2 found at:", cand)
        break
else:
    print("WARNING: arc_agi2 package not found on the candidate paths above.")

import arc_agi2
print("arc_agi2 version:", arc_agi2.__version__)

In [ ]:
# --- Locate the competition test challenges file ---------------------------
# The hidden test challenges live under /kaggle/input/<competition-slug>/.
search_roots = ["/kaggle/input", "."]
challenge_files = []
for root in search_roots:
    challenge_files += glob.glob(f"{root}/**/*test_challenges.json", recursive=True)

assert challenge_files, "Could not find *_test_challenges.json under /kaggle/input"
CHALLENGES_PATH = challenge_files[0]
print("Using challenges:", CHALLENGES_PATH)

In [ ]:
# --- Generate and write the submission -------------------------------------
from arc_agi2 import load_tasks, build_submission, write_submission

tasks = load_tasks(CHALLENGES_PATH)
print(f"Loaded {len(tasks)} tasks")

submission = build_submission(tasks)

# tasks=... enforces full task-id coverage and both attempts present.
path = write_submission(submission, "submission.json", tasks=tasks)
print("Wrote", path)

In [ ]:
# --- Sanity check the written file -----------------------------------------
written = json.loads(Path("submission.json").read_text())
assert set(written) == set(tasks), "task-id mismatch"
example_id = next(iter(written))
print("tasks in submission:", len(written))
print("example entry:", example_id, "->", json.dumps(written[example_id])[:120], "...")

## Next steps

`arc_agi2` ships a transparent, rule-based baseline (geometric, tiling,
symmetry, colour-mapping and cropping solvers). To climb the leaderboard,
add new solvers under `arc_agi2/solvers/` — each one inspects the train
pairs and returns a predictor only when it explains *every* demonstration.
The pipeline automatically ranks and de-duplicates candidates into the two
allowed attempts. Score changes locally with:

```
python -m arc_agi2 score --challenges <eval_challenges>.json --solutions <eval_solutions>.json
```